# Ilham on a free Kaggle GPU
Runs **Ollama** (her brain) on Kaggle's free T4 and exposes it through a free Cloudflare tunnel.
Copy the printed `https://...trycloudflare.com` URL into Ilham's console:
`/set ollamaurl <url>` then `/set model qwen2.5:7b`.

**Before running:** right panel → *Accelerator* = **GPU T4 x2**, *Internet* = **On** (needs a phone-verified Kaggle account). Keep this cell running while you use Ilham; stop it when done to save your weekly hours.

In [ ]:
import os, re, subprocess, time

MODEL = 'qwen2.5:7b'   # good Urdu + coding on a 16GB T4; try llama3.1:8b too

# 1) Install Ollama
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# 2) Start the Ollama server, reachable over the tunnel
env = dict(os.environ, OLLAMA_HOST='0.0.0.0:11434', OLLAMA_ORIGINS='*')
subprocess.Popen(['ollama', 'serve'], env=env)
time.sleep(6)

# 3) Download her model (first time only this session)
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('\nModel ready:', MODEL)

# 4) Free Cloudflare tunnel (no account needed)
subprocess.run('curl -sL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared', shell=True, check=True)
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 5) Print the public URL, then keep running so the server stays up
for line in tun.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n\n============================================================')
        print('  In Ilham, open the Console and paste:')
        print('     /set ollamaurl ' + m.group(0))
        print('     /set model ' + MODEL)
        print('     /model          (to confirm it is connected)')
        print('  Keep THIS cell running while you chat with her.')
        print('============================================================\n')
